In [ ]:
import sys
import os
import random
from pathlib import Path

import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pytorch_lightning as pl

In [2]:
project_root = Path.cwd().parent 
sys.path.append(str(project_root))

In [ ]:
from img_generation.models.pixel_cnn import train_pixel_cnn
from img_generation.models.vae import VQVAE, NUM_EMBEDDINGS
from img_generation.utils import show_imgs, visualize_val_reconstruction
from img_generation.datasets import Config, DatasetType, load_dataloaders, VAE_TRANSFORM

/Users/kwasus/Desktop/STUDIA/SEM6/LLM/PROJECT/Autoregressive-Image-Generation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
device

device(type='mps')

In [ ]:
config = Config(dataset_type=DatasetType.CIFAR10, data_dir="../data/cifar10", model_type="vae", batch_size=128, num_workers=4)
dataloaders, classes = load_dataloaders(config=config, transforms=VAE_TRANSFORM)

print(f"All classes: {classes}")
print(f"Number of all classes: {len(classes)}")

100%|██████████| 170M/170M [10:30<00:00, 270kB/s]    


Loaded train dataset: 45000 images, 10 classes.
All classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Number of all classes: 10


In [ ]:
model_path = "../pretrained_models/vae"

vq_vae = VQVAE()
vq_vae.load_state_dict(torch.load(f=os.path.join([model_path, "vq_vae.pth"]), weights_only=True, map_location=device))

In [ ]:
vq_vae.eval()

tokenized_dataloaders = {}

with torch.inference_mode():
    for phase in ["train", "val", "test"]:
        all_tokens = []
        for imgs, labels in dataloaders[phase]:
            imgs = imgs.to(device)
            z_q, vq_loss, encoding_indices = vq_vae.encode(imgs) # encoding_indices.shape = (b * h * w)
            b, c, h, w = z_q.shape
            tokens_2d = encoding_indices.view(b, h, w)  # from batch of inline tensor of pixel-tokens [0..8x8] to tensor-matrix of pixel-tokens [8, 8]
            all_tokens.append(tokens_2d.cpu())
        
        final_tokens = torch.cat(all_tokens, dim=0).long()
        token_dataset = TensorDataset(final_tokens, final_tokens)
        
        is_train = (phase == "train")
        tokenized_dataloaders[phase] = DataLoader(
            token_dataset,
            batch_size=config.batch_size,
            shuffle=is_train,
            drop_last=is_train,
            pin_memory=True,
            num_workers=config.num_workers
        ) 

In [ ]:
input_channels = 1  # only 1 chanel per token - token's id 
channels_hidden = 64
model_path = "../pretrained_models/pixel_cnn_on_tokens"
epochs = 70
num_classes = NUM_EMBEDDINGS

model, result = train_pixel_cnn(epochs=epochs, dataloaders=tokenized_dataloaders, device=device, model_path=model_path, c_in=input_channels, c_hidden=channels_hidden, num_classes=num_classes)


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type                       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv_vstack │ VerticalStackConvolution   │  1.8 K │ train │     0 │
│ 1 │ conv_hstack │ HorizontalStackConvolution │    640 │ train │     0 │
│ 2 │ conv_layers │ ModuleList                 │  834 K │ train │     0 │
│ 3 │ conv_out    │ Conv2d                     │  1.9 K │ train │     0 │
└───┴─────────────┴────────────────────────────┴────────┴───────┴───────┘

Trainable params: 839 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 839 K                                                                                                
Total estimated model params size (MB): 3.356                                                                      
Modules in train mode: 55                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/kwasus/Desktop/STUDIA/SEM6/LLM/PROJECT/Autoregressive-Image-Generation/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/kwasus/Desktop/STUDIA/SEM6/LLM/PROJECT/Autoregressive-Image-Generation/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


...

Epoch 4: 100%|██████████| 351/351 [03:58<00:00,  1.47it/s, v_num=0]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 351/351 [03:58<00:00,  1.47it/s, v_num=0]


/Users/kwasus/Desktop/STUDIA/SEM6/LLM/PROJECT/Autoregressive-Image-Generation/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Testing DataLoader 0: 100%|██████████| 40/40 [00:11<00:00,  3.59it/s]


In [ ]:
torch.save(model.state_dict(), f=os.path.join(model_path, "pixel_cnn.pth"))

In [ ]:
test_resl_imgs = [item[0] for item in dataloaders["test"][:20]]
test_reconstruction = result["test"][:20]
predicted_tokens = test_reconstruction["pred"]
predicted_tokens = predicted_tokens.to(device)

vq_vae.eval()
with torch.no_grad():
    b, h, w = predicted_tokens.shape
    z_q = vq_vae.vq_layer.embedding(predicted_tokens.long())
    z_q = z_q.permute(0, 3, 1, 2).contiguous()  # decoder // DeconvBlock takes [batch_size, channels, height, width]
    reconstructed_images = vq_vae.normalize(vq_vae.decoder(z_q))


visualize_val_reconstruction(test_resl_imgs, reconstructed_images)

In [ ]:
num_samples = 24
pl.seed_everything(1)

samples = model.sample(img_shape=(num_samples, input_channels, h, w))   # sample tokens of shape [B, Ch=1, H, W], here H=8, W=8
samples = samples.squeeze(1) # [B, Ch=1, H, W] --> [B, H, W]
samples = samples.to(device)

vq_vae.eval()
with torch.no_grad():
    b, h, w = samples.shape
    z_q = vq_vae.vq_layer.embedding(samples.long())
    z_q = z_q.permute(0, 3, 1, 2).contiguous()  # decoder // DeconvBlock takes [B, Ch, H, W]
    generated_images = vq_vae.normalize(vq_vae.decoder(z_q))

show_imgs(generated_images.cpu())